In [ ]:
# ============================================================
# Hybrid Graph-Text Architecture for Sports Match Outcome Prediction
# GNN Stage
# ============================================================

# Цель:
# 0 = home_win
# 1 = draw
# 2 = away_win
#
# Основной modeling subset:
# train: 1994-2014
# val:   2018
# test:  2022
#
# В этом notebook:
# - загружаем model-ready данные
# - загружаем graph nodes / edges
# - проверяем graph schema
# - готовим match node features: numeric + text PCA
# - строим simplified Team-Match GNN prototype
# - обучаем PyTorch Geometric модель, если доступна
# - сохраняем predictions и gnn_results.csv
# - сравниваем с baseline models


# Imports, paths, reproducibility

In [2]:
import os
import json
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Ограничение потоков — полезно для macOS и small-data экспериментов
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
os.environ["NUMEXPR_NUM_THREADS"] = "2"

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    TORCH_AVAILABLE = True
    
    torch.set_num_threads(2)
except Exception as e:
    TORCH_AVAILABLE = False
    print("PyTorch import failed:", repr(e))

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

RANDOM_SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    if TORCH_AVAILABLE:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(RANDOM_SEED)

DATA_DIR = Path("SNA/data/processed")
DATA_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cpu"
if TORCH_AVAILABLE and torch.cuda.is_available():
    DEVICE = "cuda"

print("TORCH_AVAILABLE:", TORCH_AVAILABLE)
print("DEVICE:", DEVICE)
print("DATA_DIR:", DATA_DIR.resolve())


TORCH_AVAILABLE: True
DEVICE: cpu
DATA_DIR: /Users/gaperov/Documents/University/SNA/data/processed


# Проверка PyTorch Geometric

In [3]:
PYG_AVAILABLE = False

try:
    import torch_geometric
    from torch_geometric.data import HeteroData
    from torch_geometric.nn import HeteroConv, SAGEConv, Linear
    PYG_AVAILABLE = True
    print("torch_geometric available:", torch_geometric.__version__)
except Exception as e:
    print("torch_geometric is not available.")
    print("Reason:", repr(e))
    print()
    print("Install suggestion:")
    print("  pip install torch-geometric")
    print()
    print("For macOS / Python 3.14, if installation fails, consider:")
    print("  - Python 3.11 or 3.12 virtualenv")
    print("  - CPU-only PyTorch")
    print("  - then pip install torch-geometric")
    PYG_AVAILABLE = False


torch_geometric available: 2.7.0


# Вспомогательные функции загрузки

In [4]:
def read_table_auto(path_without_ext_or_path):
    """
    Читает parquet/csv.
    Можно передать:
    - Path("file.parquet")
    - Path("file.csv")
    - Path("file") без расширения
    """
    p = Path(path_without_ext_or_path)
    
    candidates = []
    if p.suffix:
        candidates.append(p)
    else:
        candidates.extend([
            p.with_suffix(".parquet"),
            p.with_suffix(".csv")
        ])
    
    for c in candidates:
        if c.exists():
            if c.suffix == ".parquet":
                return pd.read_parquet(c)
            elif c.suffix == ".csv":
                return pd.read_csv(c)
            else:
                raise ValueError(f"Unsupported extension: {c}")
    
    raise FileNotFoundError(f"No file found for {path_without_ext_or_path}. Tried: {candidates}")


def read_json_if_exists(path, default=None):
    path = Path(path)
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return default


def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


# Загрузка model-ready матчей

In [5]:
match_path_base = DATA_DIR / "match_dataset_model_ready"

matches = read_table_auto(match_path_base)

print("matches shape:", matches.shape)
print("columns:", matches.columns.tolist()[:30], "...")

required_cols = ["match_id", "tournament_year", "target"]
missing_required = [c for c in required_cols if c not in matches.columns]
if missing_required:
    raise ValueError(f"Missing required columns in matches: {missing_required}")

print()
print("Tournament years:", sorted(matches["tournament_year"].dropna().unique().tolist()))
print()
print("Target distribution overall:")
display(matches["target"].value_counts(normalize=False).sort_index())
display(matches["target"].value_counts(normalize=True).sort_index())


matches shape: (500, 127)
columns: ['tournament_year', 'match_date', 'stage', 'home_team_name', 'away_team_name', 'home_team_norm', 'away_team_norm', 'home_score', 'away_score', 'stadium_name', 'city', 'referee_name', 'attendance', 'home_manager_name', 'away_manager_name', 'tournament_id', 'match_id', 'home_team_id', 'away_team_id', 'stadium_id', 'referee_id', 'home_manager_id', 'away_manager_id', 'result', 'target', 'home_fifa_rank', 'home_fifa_points', 'home_fifa_previous_points', 'home_fifa_diff_points', 'home_fifa_ranking_date'] ...

Tournament years: [1994, 1998, 2002, 2006, 2010, 2014, 2018, 2022]

Target distribution overall:


target
0    219
1    118
2    163
Name: count, dtype: int64

target
0    0.438
1    0.236
2    0.326
Name: proportion, dtype: float64

# Загрузка или создание train/val/test split

In [6]:
train_path_base = DATA_DIR / "train_matches_model_ready"
val_path_base = DATA_DIR / "val_matches_model_ready"
test_path_base = DATA_DIR / "test_matches_model_ready"

try:
    train_df = read_table_auto(train_path_base)
    val_df = read_table_auto(val_path_base)
    test_df = read_table_auto(test_path_base)
    print("Loaded existing split files.")
except FileNotFoundError:
    print("Split files not found. Creating split by tournament_year.")
    
    train_years = [1994, 1998, 2002, 2006, 2010, 2014]
    val_years = [2018]
    test_years = [2022]
    
    train_df = matches[matches["tournament_year"].isin(train_years)].copy()
    val_df = matches[matches["tournament_year"].isin(val_years)].copy()
    test_df = matches[matches["tournament_year"].isin(test_years)].copy()
    
    # Сохраняем для воспроизводимости
    train_df.to_parquet(DATA_DIR / "train_matches_model_ready.parquet", index=False)
    val_df.to_parquet(DATA_DIR / "val_matches_model_ready.parquet", index=False)
    test_df.to_parquet(DATA_DIR / "test_matches_model_ready.parquet", index=False)

print("train shape:", train_df.shape)
print("val shape:", val_df.shape)
print("test shape:", test_df.shape)

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"\n{name} years:", sorted(df["tournament_year"].unique().tolist()))
    print(f"{name} target distribution:")
    print(df["target"].value_counts(normalize=False).sort_index())
    print(df["target"].value_counts(normalize=True).sort_index())


Loaded existing split files.
train shape: (372, 127)
val shape: (64, 127)
test shape: (64, 127)

train years: [1994, 1998, 2002, 2006, 2010, 2014]
train target distribution:
target
0    164
1     90
2    118
Name: count, dtype: int64
target
0    0.440860
1    0.241935
2    0.317204
Name: proportion, dtype: float64

val years: [2018]
val target distribution:
target
0    26
1    13
2    25
Name: count, dtype: int64
target
0    0.406250
1    0.203125
2    0.390625
Name: proportion, dtype: float64

test years: [2022]
test target distribution:
target
0    29
1    15
2    20
Name: count, dtype: int64
target
0    0.453125
1    0.234375
2    0.312500
Name: proportion, dtype: float64


# Загрузка feature groups и class weights

In [44]:
feature_groups_path = DATA_DIR / "feature_groups.json"
class_weights_path = DATA_DIR / "class_weights.json"

feature_groups = read_json_if_exists(feature_groups_path, default=None)

if feature_groups is None:
    print("feature_groups.json not found. Reconstructing basic feature groups from columns.")
    
    leakage_cols = {
        "home_score", "away_score", "result", "target",
        "home_xg", "away_xg", "xg"
    }
    
    non_feature_cols = {
        "match_id", "tournament_id", "tournament_year",
        "home_team_name", "away_team_name",
        "match_date", "date",
        "combined_pre_match_text", "sources",
        "latest_text_time", "earliest_text_time"
    }
    
    numeric_cols_auto = []
    for c in matches.columns:
        if c in leakage_cols or c in non_feature_cols:
            continue
        if pd.api.types.is_numeric_dtype(matches[c]):
            numeric_cols_auto.append(c)
    
    feature_groups = {
        "combined_numeric": numeric_cols_auto
    }
    
    save_json(feature_groups, feature_groups_path)
else:
    print("Loaded feature_groups.json")

print("Feature groups:")
for k, v in feature_groups.items():
    print(k, len(v) if isinstance(v, list) else type(v))

class_weights_obj = read_json_if_exists(class_weights_path, default=None)

if class_weights_obj is None:
    print("class_weights.json not found. Using recommended weights.")
    class_weights_obj = {
        "0": 1.00,
        "1": 1.86,
        "2": 1.35
    }
    save_json(class_weights_obj, class_weights_path)
else:
    print("Loaded class_weights.json")

class_weights = {int(k): float(v) for k, v in class_weights_obj['eda_recommended'].items()}
print("class_weights:", class_weights)


Loaded feature_groups.json
Feature groups:
ranking_full 15
ranking_compact 3
form_last10 12
form_last5 12
form_last3 12
wc_history 14
tournament_cumulative 10
rest 4
text_meta 7
stage 7
combined_numeric 69
Loaded class_weights.json
class_weights: {0: 1.0, 1: 1.86, 2: 1.35}


# Загрузка text embeddings

In [45]:
text_emb_path = DATA_DIR / "match_text_embeddings_allMiniLM.npy"
text_index_path = DATA_DIR / "match_text_embeddings_index.csv"

if not text_emb_path.exists():
    raise FileNotFoundError(f"Missing text embeddings file: {text_emb_path}")

if not text_index_path.exists():
    raise FileNotFoundError(f"Missing text embedding index file: {text_index_path}")

text_embeddings = np.load(text_emb_path)
text_index = pd.read_csv(text_index_path)

print("text_embeddings shape:", text_embeddings.shape)
print("text_index shape:", text_index.shape)
display(text_index.head())

required_text_index_cols = ["match_id", "embedding_row"]
missing_text_cols = [c for c in required_text_index_cols if c not in text_index.columns]
if missing_text_cols:
    raise ValueError(f"Missing required columns in text_index: {missing_text_cols}")

# Coverage check
match_ids = set(matches["match_id"].astype(str))
text_ids = set(text_index["match_id"].astype(str))
print("Match ids:", len(match_ids))
print("Text index ids:", len(text_ids))
print("Coverage match -> text:", len(match_ids & text_ids), "/", len(match_ids))
print("Missing text index for matches:", len(match_ids - text_ids))

if "text_available" in text_index.columns:
    print("text_available distribution in text_index:")
    display(text_index["text_available"].value_counts(dropna=False))


text_embeddings shape: (500, 384)
text_index shape: (500, 6)


,match_id,tournament_year,home_team_name,away_team_name,text_available,embedding_row
0,match_24ab401684d2,1994,Germany,Bolivia,0,0
1,match_38ef6f79f047,1994,Spain,Korea Republic,0,1
2,match_ebe85d2cc918,1994,Colombia,Romania,0,2
3,match_f0b250615ce7,1994,Italy,Republic of Ireland,0,3
4,match_fd7cb15508dc,1994,United States,Switzerland,0,4


Match ids: 500
Text index ids: 500
Coverage match -> text: 500 / 500
Missing text index for matches: 0
text_available distribution in text_index:


text_available
1    321
0    179
Name: count, dtype: int64

# Загрузка graph nodes / edges

In [46]:
graph_nodes = read_table_auto(DATA_DIR / "graph_nodes_model_ready")
graph_edges = read_table_auto(DATA_DIR / "graph_edges_model_ready")

print("graph_nodes shape:", graph_nodes.shape)
print("graph_edges shape:", graph_edges.shape)

print("\ngraph_nodes columns:")
print(graph_nodes.columns.tolist())

print("\ngraph_edges columns:")
print(graph_edges.columns.tolist())

display(graph_nodes.head())
display(graph_edges.head())


graph_nodes shape: (1007, 7)
graph_edges shape: (4741, 10)

graph_nodes columns:
['node_id', 'node_type', 'name', 'canonical_name', 'source_id', 'tournament_year', 'metadata_json']

graph_edges columns:
['edge_id', 'source_node_id', 'target_node_id', 'edge_type', 'match_id', 'tournament_id', 'tournament_year', 'timestamp', 'weight', 'metadata_json']


,node_id,node_type,name,canonical_name,source_id,tournament_year,metadata_json
0,team_28ef36b35ae8,Team,Germany,germany,NaN,NaN,"{""team_norm"": ""germany""}"
1,team_1747d4b2dd4b,Team,Spain,spain,NaN,NaN,"{""team_norm"": ""spain""}"
2,team_25446782e2cc,Team,Colombia,colombia,NaN,NaN,"{""team_norm"": ""colombia""}"
3,team_bab4ac375ada,Team,Italy,italy,NaN,NaN,"{""team_norm"": ""italy""}"
4,team_152649df347e,Team,United States,united states,NaN,NaN,"{""team_norm"": ""united states""}"


,edge_id,source_node_id,target_node_id,edge_type,match_id,tournament_id,tournament_year,timestamp,weight,metadata_json
0,edge_8cdaebc5d11a,match_24ab401684d2,wc_1994,match_belongs_to_tournament,match_24ab401684d2,wc_1994,1994,1994-06-17,1.0,{}
1,edge_c8b436f81868,match_24ab401684d2,stadium_21a36eff12d1,match_played_at_stadium,match_24ab401684d2,wc_1994,1994,1994-06-17,1.0,{}
2,edge_e7376ce96f9e,referee_082a13461fbe,match_24ab401684d2,referee_officiated_match,match_24ab401684d2,wc_1994,1994,1994-06-17,1.0,{}
3,edge_cee57db5b986,team_28ef36b35ae8,match_24ab401684d2,team_played_match,match_24ab401684d2,wc_1994,1994,1994-06-17,1.0,"{""side"": ""home"", ""team_fifa_rank"": 3.0, ""team_..."
4,edge_bf0ca135aaf5,team_28ef36b35ae8,team_76781be1c79d,team_played_against_team,match_24ab401684d2,wc_1994,1994,1994-06-17,1.0,"{""side"": ""home"", ""stage_norm"": ""group"", ""is_kn..."


# Проверка graph schema

In [47]:
# Определяем вероятные имена ключевых колонок
node_id_col_candidates = ["node_id", "graph_node_id", "id"]
node_type_col_candidates = ["node_type", "type", "entity_type"]

edge_src_col = "source_node_id"
edge_dst_col = "target_node_id"
edge_type_col = "edge_type"

node_id_col = next((c for c in node_id_col_candidates if c in graph_nodes.columns), None)
node_type_col = next((c for c in node_type_col_candidates if c in graph_nodes.columns), None)

if node_id_col is None:
    raise ValueError(f"Cannot find node id column. Tried {node_id_col_candidates}")

if node_type_col is None:
    raise ValueError(f"Cannot find node type column. Tried {node_type_col_candidates}")

for c in [edge_src_col, edge_dst_col, edge_type_col]:
    if c not in graph_edges.columns:
        raise ValueError(f"Missing required edge column: {c}")

print("node_id_col:", node_id_col)
print("node_type_col:", node_type_col)

print("\nNode types:")
display(graph_nodes[node_type_col].value_counts(dropna=False))

print("\nEdge types:")
display(graph_edges[edge_type_col].value_counts(dropna=False))

# Проверка match_id coverage в graph_edges
if "match_id" in graph_edges.columns:
    edge_match_ids = set(graph_edges["match_id"].dropna().astype(str))
    print("\nEdge match_id coverage:")
    print("edge match ids:", len(edge_match_ids))
    print("dataset match ids:", len(match_ids))
    print("intersection:", len(edge_match_ids & match_ids))
    print("matches missing in edge.match_id:", len(match_ids - edge_match_ids))
else:
    print("graph_edges has no match_id column")

# Проверка Match nodes
match_node_candidates = graph_nodes[
    graph_nodes[node_type_col].astype(str).str.lower().isin(["match", "matches"])
].copy()

print("\nMatch nodes found:", match_node_candidates.shape)

# Ищем match_id в graph_nodes
possible_match_id_cols = ["match_id", "entity_id", "original_id", "source_id"]
available_match_id_cols = [c for c in possible_match_id_cols if c in graph_nodes.columns]
print("Possible match id cols in graph_nodes:", available_match_id_cols)

for c in available_match_id_cols:
    vals = set(match_node_candidates[c].dropna().astype(str))
    print(f"Coverage using graph_nodes[{c}]:", len(vals & match_ids), "/", len(match_ids))


node_id_col: node_id
node_type_col: node_type

Node types:


node_type
Match             500
TeamTournament    248
Stadium            93
Referee            88
Team               70
Tournament          8
Name: count, dtype: int64


Edge types:


edge_type
team_played_match                  1000
team_played_against_team           1000
team_tournament_played_match       1000
match_belongs_to_tournament         500
match_played_at_stadium             500
team_participated_in_tournament     248
team_has_tournament_instance        248
referee_officiated_match            245
Name: count, dtype: int64


Edge match_id coverage:
edge match ids: 500
dataset match ids: 500
intersection: 500
matches missing in edge.match_id: 0

Match nodes found: (500, 7)
Possible match id cols in graph_nodes: ['source_id']
Coverage using graph_nodes[source_id]: 500 / 500


# Загрузка baseline results

In [48]:
baseline_results_path = DATA_DIR / "model_results_ablation.csv"

if baseline_results_path.exists():
    baseline_results = pd.read_csv(baseline_results_path)
    print("Loaded baseline results:", baseline_results.shape)
    display(baseline_results.head())
else:
    baseline_results = pd.DataFrame()
    print("Baseline results file not found:", baseline_results_path)


Loaded baseline results: (29, 14)


,model_name,feature_set,accuracy_val,balanced_accuracy_val,macro_f1_val,weighted_f1_val,draw_recall_val,draw_f1_val,accuracy_test,balanced_accuracy_test,macro_f1_test,weighted_f1_test,draw_recall_test,draw_f1_test
0,ranking_compact_rf,ranking_compact,0.593750,0.573846,0.565187,0.596776,0.461538,0.400000,0.500000,0.477969,0.469039,0.497771,0.266667,0.275862
1,numeric_text_rf,combined_numeric_plus_text_embeddings,0.640625,0.562051,0.546268,0.606456,0.153846,0.235294,0.578125,0.551724,0.553846,0.576923,0.400000,0.461538
2,combined_numeric_rf_safe,combined_numeric,0.531250,0.483077,0.489872,0.547501,0.230769,0.193548,0.546875,0.550192,0.538936,0.552258,0.533333,0.457143
3,ranking_full_rf,ranking_full,0.531250,0.469744,0.480545,0.548777,0.153846,0.133333,0.500000,0.500192,0.492308,0.511418,0.533333,0.400000
4,combined_numeric_gb_safe,combined_numeric,0.546875,0.482564,0.480503,0.543113,0.153846,0.160000,0.546875,0.518391,0.517921,0.549059,0.400000,0.387097


# Выбор numeric features для Match nodes

In [49]:
LEAKAGE_COLS = {
    "home_score", "away_score", "result", "target",
    "home_xg", "away_xg", "xg"
}

ID_TIME_TEXT_COLS = {
    "match_id", "tournament_id", "tournament_year",
    "home_team_name", "away_team_name",
    "home_team_id", "away_team_id",
    "match_date", "date", "kickoff",
    "combined_pre_match_text", "sources",
    "latest_text_time", "earliest_text_time"
}

if "combined_numeric" in feature_groups:
    candidate_numeric = list(feature_groups["combined_numeric"])
elif "selected_numeric" in feature_groups:
    candidate_numeric = list(feature_groups["selected_numeric"])
else:
    candidate_numeric = [
        c for c in matches.columns
        if pd.api.types.is_numeric_dtype(matches[c])
    ]

numeric_features = []
for c in candidate_numeric:
    if c in LEAKAGE_COLS:
        continue
    if c in ID_TIME_TEXT_COLS:
        continue
    if c not in matches.columns:
        continue
    if not pd.api.types.is_numeric_dtype(matches[c]):
        continue
    numeric_features.append(c)

# Дополнительно исключаем obvious target-like колонки по имени
bad_substrings = ["score", "xg", "result", "target"]
numeric_features = [
    c for c in numeric_features
    if not any(bs in c.lower() for bs in bad_substrings)
]

print("Number of numeric_features:", len(numeric_features))
print(numeric_features)


Number of numeric_features: 69
['away_fifa_diff_points', 'away_fifa_points', 'away_fifa_rank', 'away_first_match_in_tournament', 'away_last10_goal_diff', 'away_last10_goals_against', 'away_last10_goals_for', 'away_last10_matches', 'away_last10_points', 'away_last10_win_rate', 'away_rest_days', 'away_tournament_goal_diff_before', 'away_tournament_goals_against_before', 'away_tournament_goals_for_before', 'away_tournament_matches_before', 'away_tournament_points_before', 'away_wc_goal_diff_before', 'away_wc_goals_against_before', 'away_wc_goals_for_before', 'away_wc_matches_before', 'away_wc_points_before', 'away_wc_win_rate_before', 'away_wc_wins_before', 'fifa_momentum_diff', 'fifa_momentum_diff_home_minus_away', 'fifa_points_diff', 'fifa_points_diff_home_minus_away', 'fifa_rank_diff', 'fifa_rank_diff_away_minus_home', 'fifa_rank_diff_home_minus_away', 'fifa_strength_pc1', 'guardian_text_flag', 'home_better_rank_flag', 'home_fifa_diff_points', 'home_fifa_points', 'home_fifa_rank', 'hom

# Подготовка numeric matrix: imputer + scaler fit только на train

In [50]:
# Упорядочиваем matches для построения graph features
matches_all = matches.copy()
matches_all["match_id_str"] = matches_all["match_id"].astype(str)

train_ids = set(train_df["match_id"].astype(str))
val_ids = set(val_df["match_id"].astype(str))
test_ids = set(test_df["match_id"].astype(str))

train_mask_rows = matches_all["match_id_str"].isin(train_ids).values
val_mask_rows = matches_all["match_id_str"].isin(val_ids).values
test_mask_rows = matches_all["match_id_str"].isin(test_ids).values

X_num_raw = matches_all[numeric_features].copy()

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_train_num_raw = X_num_raw.loc[train_mask_rows]

X_train_imp = imputer.fit_transform(X_train_num_raw)
scaler.fit(X_train_imp)

X_num_imp = imputer.transform(X_num_raw)
X_num_scaled = scaler.transform(X_num_imp).astype(np.float32)

print("X_num_scaled:", X_num_scaled.shape)
print("NaNs:", np.isnan(X_num_scaled).sum())


X_num_scaled: (500, 69)
NaNs: 0


# Подготовка text embeddings и PCA fit только на train

In [129]:
TEXT_PCA_DIM = 64  # можно попробовать 64, но для small-data 32 часто стабильнее

# Создаём mapping match_id -> embedding_row
text_index = text_index.copy()
text_index["match_id_str"] = text_index["match_id"].astype(str)
embedding_row_map = dict(zip(text_index["match_id_str"], text_index["embedding_row"]))

emb_dim = text_embeddings.shape[1]
X_text = np.zeros((len(matches_all), emb_dim), dtype=np.float32)

missing_text_index_count = 0

for i, mid in enumerate(matches_all["match_id_str"].values):
    if mid in embedding_row_map:
        row = int(embedding_row_map[mid])
        if 0 <= row < len(text_embeddings):
            X_text[i] = text_embeddings[row].astype(np.float32)
        else:
            missing_text_index_count += 1
    else:
        missing_text_index_count += 1

print("X_text:", X_text.shape)
print("Missing text index count:", missing_text_index_count)

# PCA fit только на train
if TEXT_PCA_DIM is not None and TEXT_PCA_DIM > 0 and TEXT_PCA_DIM < emb_dim:
    pca = PCA(n_components=TEXT_PCA_DIM, random_state=RANDOM_SEED)
    pca.fit(X_text[train_mask_rows])
    X_text_pca = pca.transform(X_text).astype(np.float32)
    print("X_text_pca:", X_text_pca.shape)
    print("Explained variance ratio sum:", float(pca.explained_variance_ratio_.sum()))
else:
    pca = None
    X_text_pca = X_text.astype(np.float32)
    print("Using raw text embeddings:", X_text_pca.shape)


X_text: (500, 384)
Missing text index count: 0
X_text_pca: (500, 64)
Explained variance ratio sum: 0.8806828260421753


# Text meta и stage features

In [130]:
extra_feature_candidates = [
    "text_available",
    "log_text_count",
    "text_count",
    "guardian_text_flag",
    "no_text_flag"
]

# Stage features — если уже one-hot/numeric
stage_candidates = [
    c for c in matches_all.columns
    if ("stage" in c.lower() or "round" in c.lower())
    and pd.api.types.is_numeric_dtype(matches_all[c])
    and c not in LEAKAGE_COLS
]

extra_features = [
    c for c in extra_feature_candidates
    if c in matches_all.columns and pd.api.types.is_numeric_dtype(matches_all[c])
]

print("extra text features:", extra_features)
print("stage numeric candidates:", stage_candidates)

# Для базового prototype добавим text meta, stage можно включать отдельной ablation
USE_STAGE_FEATURES = False

extra_cols = extra_features + (stage_candidates if USE_STAGE_FEATURES else [])

if extra_cols:
    extra_imputer = SimpleImputer(strategy="median")
    extra_scaler = StandardScaler()
    
    X_extra_raw = matches_all[extra_cols].copy()
    X_extra_train_imp = extra_imputer.fit_transform(X_extra_raw.loc[train_mask_rows])
    extra_scaler.fit(X_extra_train_imp)
    
    X_extra = extra_scaler.transform(extra_imputer.transform(X_extra_raw)).astype(np.float32)
else:
    X_extra = np.zeros((len(matches_all), 0), dtype=np.float32)

print("X_extra:", X_extra.shape)


extra text features: ['text_available', 'log_text_count', 'text_count', 'guardian_text_flag', 'no_text_flag']
stage numeric candidates: ['stage_final', 'stage_group', 'stage_quarter_final', 'stage_round_of_16', 'stage_semi_final', 'stage_third_place']
X_extra: (500, 5)


# Финальная матрица Match node features

In [131]:
X_match_numeric_only = X_num_scaled.astype(np.float32)
X_match_numeric_text = np.concatenate([X_num_scaled, X_text_pca], axis=1).astype(np.float32)
X_match_numeric_text_extra = np.concatenate([X_num_scaled, X_text_pca, X_extra], axis=1).astype(np.float32)

print("X_match_numeric_only:", X_match_numeric_only.shape)
print("X_match_numeric_text:", X_match_numeric_text.shape)
print("X_match_numeric_text_extra:", X_match_numeric_text_extra.shape)


X_match_numeric_only: (500, 69)
X_match_numeric_text: (500, 133)
X_match_numeric_text_extra: (500, 138)


# Стратегия Team node features

In [18]:
# Strategy:
#
# Для первого transductive prototype:
# - Team nodes получают простые structural features:
#   - normalized degree within constructed graph
#   - constant 1
# - Плюс модель может иметь learnable transformation для Team nodes.
#
# Почему не используем latest FIFA rank как static team feature:
# - для команды rank меняется во времени;
# - static feature, агрегированный по всем годам, может содержать future information.
#
# Leakage-safer дальнейшая стратегия:
# - для каждого матча использовать team snapshot features only before match_date;
# - либо строить dynamic graph / temporal message passing;
# - либо использовать TeamTournament nodes, где признаки рассчитаны до начала турнира.


# Извлечение Team и Match nodes из graph_nodes

In [132]:
nodes = graph_nodes.copy()
edges = graph_edges.copy()

nodes[node_id_col] = nodes[node_id_col].astype(str)
edges[edge_src_col] = edges[edge_src_col].astype(str)
edges[edge_dst_col] = edges[edge_dst_col].astype(str)

nodes["_node_type_lower"] = nodes[node_type_col].astype(str).str.lower()

team_type_names = {"team", "teams"}
match_type_names = {"match", "matches"}

team_nodes = nodes[nodes["_node_type_lower"].isin(team_type_names)].copy()
match_nodes_graph = nodes[nodes["_node_type_lower"].isin(match_type_names)].copy()

print("team_nodes:", team_nodes.shape)
print("match_nodes_graph:", match_nodes_graph.shape)

display(team_nodes.head())
display(match_nodes_graph.head())


team_nodes: (70, 8)
match_nodes_graph: (500, 8)


,node_id,node_type,name,canonical_name,source_id,tournament_year,metadata_json,_node_type_lower
0,team_28ef36b35ae8,Team,Germany,germany,NaN,NaN,"{""team_norm"": ""germany""}",team
1,team_1747d4b2dd4b,Team,Spain,spain,NaN,NaN,"{""team_norm"": ""spain""}",team
2,team_25446782e2cc,Team,Colombia,colombia,NaN,NaN,"{""team_norm"": ""colombia""}",team
3,team_bab4ac375ada,Team,Italy,italy,NaN,NaN,"{""team_norm"": ""italy""}",team
4,team_152649df347e,Team,United States,united states,NaN,NaN,"{""team_norm"": ""united states""}",team


,node_id,node_type,name,canonical_name,source_id,tournament_year,metadata_json,_node_type_lower
70,match_24ab401684d2,Match,Germany vs Bolivia 1994,match_24ab401684d2,match_24ab401684d2,1994.0,"{""match_date"": ""1994-06-17 00:00:00"", ""stage"":...",match
71,match_38ef6f79f047,Match,Spain vs Korea Republic 1994,match_38ef6f79f047,match_38ef6f79f047,1994.0,"{""match_date"": ""1994-06-17 00:00:00"", ""stage"":...",match
72,match_ebe85d2cc918,Match,Colombia vs Romania 1994,match_ebe85d2cc918,match_ebe85d2cc918,1994.0,"{""match_date"": ""1994-06-18 00:00:00"", ""stage"":...",match
73,match_f0b250615ce7,Match,Italy vs Republic of Ireland 1994,match_f0b250615ce7,match_f0b250615ce7,1994.0,"{""match_date"": ""1994-06-18 00:00:00"", ""stage"":...",match
74,match_fd7cb15508dc,Match,United States vs Switzerland 1994,match_fd7cb15508dc,match_fd7cb15508dc,1994.0,"{""match_date"": ""1994-06-18 00:00:00"", ""stage"":...",match


# Определение match_id для Match nodes

In [133]:
def infer_match_id_col(match_nodes_graph, dataset_match_ids):
    candidates = ["match_id", "entity_id", "original_id", "source_id"]
    best_col = None
    best_cov = -1
    
    dataset_match_ids = set(str(x) for x in dataset_match_ids)
    
    for c in candidates:
        if c in match_nodes_graph.columns:
            vals = set(match_nodes_graph[c].dropna().astype(str))
            cov = len(vals & dataset_match_ids)
            print(f"Candidate {c}: coverage {cov}/{len(dataset_match_ids)}")
            if cov > best_cov:
                best_cov = cov
                best_col = c
    
    return best_col, best_cov

match_id_col_in_nodes, match_cov = infer_match_id_col(match_nodes_graph, matches_all["match_id_str"])

if match_id_col_in_nodes is None or match_cov == 0:
    raise ValueError(
        "Cannot infer match_id column in graph_nodes for Match nodes. "
        "Check graph_nodes schema."
    )

print("Selected match_id_col_in_nodes:", match_id_col_in_nodes, "coverage:", match_cov)


Candidate source_id: coverage 500/500
Selected match_id_col_in_nodes: source_id coverage: 500


# Фильтрация Match nodes до modeling matches 1994–2022

In [134]:
match_nodes_graph["match_id_str"] = match_nodes_graph[match_id_col_in_nodes].astype(str)

model_match_ids = set(matches_all["match_id_str"])

match_nodes_model = match_nodes_graph[
    match_nodes_graph["match_id_str"].isin(model_match_ids)
].copy()

print("Match nodes in modeling dataset:", match_nodes_model.shape)

# Создаём порядок Match nodes такой же, как matches_all, если есть node
match_id_to_graph_node_id = dict(
    zip(match_nodes_model["match_id_str"], match_nodes_model[node_id_col].astype(str))
)

covered_match_ids = set(match_id_to_graph_node_id.keys())
print("Covered matches:", len(covered_match_ids), "/", len(model_match_ids))
print("Missing matches in graph:", len(model_match_ids - covered_match_ids))

# Для GNN prototype будем использовать только covered matches
matches_gnn = matches_all[matches_all["match_id_str"].isin(covered_match_ids)].copy()
matches_gnn = matches_gnn.reset_index(drop=True)

print("matches_gnn:", matches_gnn.shape)


Match nodes in modeling dataset: (500, 9)
Covered matches: 500 / 500
Missing matches in graph: 0
matches_gnn: (500, 128)


# Пересборка feature matrices под covered Match nodes

In [135]:
# Индекс original matches_all row by match_id
match_id_to_row_idx = {
    mid: i for i, mid in enumerate(matches_all["match_id_str"].values)
}

gnn_row_indices = [match_id_to_row_idx[mid] for mid in matches_gnn["match_id_str"].values]

X_match_numeric_only_gnn = X_match_numeric_only[gnn_row_indices]
X_match_numeric_text_gnn = X_match_numeric_text[gnn_row_indices]
X_match_numeric_text_extra_gnn = X_match_numeric_text_extra[gnn_row_indices]

y_gnn = matches_gnn["target"].astype(int).values

train_mask_gnn = matches_gnn["match_id_str"].isin(train_ids).values
val_mask_gnn = matches_gnn["match_id_str"].isin(val_ids).values
test_mask_gnn = matches_gnn["match_id_str"].isin(test_ids).values

print("X_match_numeric_only_gnn:", X_match_numeric_only_gnn.shape)
print("X_match_numeric_text_gnn:", X_match_numeric_text_gnn.shape)
print("y_gnn:", y_gnn.shape)
print("train/val/test:", train_mask_gnn.sum(), val_mask_gnn.sum(), test_mask_gnn.sum())

print("Target distribution in GNN covered matches:")
display(pd.Series(y_gnn).value_counts().sort_index())


X_match_numeric_only_gnn: (500, 69)
X_match_numeric_text_gnn: (500, 133)
y_gnn: (500,)
train/val/test: 372 64 64
Target distribution in GNN covered matches:


0    219
1    118
2    163
Name: count, dtype: int64

# Team node mapping и structural features

In [136]:
team_node_ids = team_nodes[node_id_col].astype(str).tolist()
team_node_id_to_idx = {nid: i for i, nid in enumerate(team_node_ids)}

match_graph_node_ids = [
    match_id_to_graph_node_id[mid]
    for mid in matches_gnn["match_id_str"].values
]
match_graph_node_id_to_idx = {
    nid: i for i, nid in enumerate(match_graph_node_ids)
}

print("num team nodes:", len(team_node_ids))
print("num match nodes:", len(match_graph_node_ids))

# Degree по всем edges prototype graph
all_relevant_node_ids = set(team_node_ids) | set(match_graph_node_ids)
degree_counter = {nid: 0 for nid in all_relevant_node_ids}

for s, t in zip(edges[edge_src_col].values, edges[edge_dst_col].values):
    if s in degree_counter:
        degree_counter[s] += 1
    if t in degree_counter:
        degree_counter[t] += 1

team_degrees = np.array([degree_counter[nid] for nid in team_node_ids], dtype=np.float32)
if team_degrees.std() > 0:
    team_degrees_norm = (team_degrees - team_degrees.mean()) / (team_degrees.std() + 1e-8)
else:
    team_degrees_norm = np.zeros_like(team_degrees)

X_team = np.stack([
    np.ones_like(team_degrees_norm),
    team_degrees_norm
], axis=1).astype(np.float32)

print("X_team:", X_team.shape)
print("team degree min/mean/max:", team_degrees.min(), team_degrees.mean(), team_degrees.max())


num team nodes: 70
num match nodes: 500
X_team: (70, 2)
team degree min/mean/max: 11.0 49.942856 160.0


# Построение simplified Team-Match edge_index

In [137]:
def build_team_match_edges(edges):
    """
    Возвращает:
    - team_to_match edge_index
    - match_to_team edge_index
    - optional team_to_team edge_index
    """
    team_to_match_pairs = []
    match_to_team_pairs = []
    team_to_team_pairs = []
    
    for _, row in edges.iterrows():
        src = str(row[edge_src_col])
        dst = str(row[edge_dst_col])
        etype = str(row[edge_type_col])
        
        src_is_team = src in team_node_id_to_idx
        dst_is_team = dst in team_node_id_to_idx
        src_is_match = src in match_graph_node_id_to_idx
        dst_is_match = dst in match_graph_node_id_to_idx
        
        # Любые связи Team-Match
        if src_is_team and dst_is_match:
            team_to_match_pairs.append([
                team_node_id_to_idx[src],
                match_graph_node_id_to_idx[dst]
            ])
            match_to_team_pairs.append([
                match_graph_node_id_to_idx[dst],
                team_node_id_to_idx[src]
            ])
        
        elif src_is_match and dst_is_team:
            match_to_team_pairs.append([
                match_graph_node_id_to_idx[src],
                team_node_id_to_idx[dst]
            ])
            team_to_match_pairs.append([
                team_node_id_to_idx[dst],
                match_graph_node_id_to_idx[src]
            ])
        
        # Team-Team rivalry / played-against
        elif src_is_team and dst_is_team:
            if "against" in etype.lower() or "played" in etype.lower():
                team_to_team_pairs.append([
                    team_node_id_to_idx[src],
                    team_node_id_to_idx[dst]
                ])
    
    def to_edge_index(pairs):
        if len(pairs) == 0:
            return np.zeros((2, 0), dtype=np.int64)
        return np.array(pairs, dtype=np.int64).T
    
    return (
        to_edge_index(team_to_match_pairs),
        to_edge_index(match_to_team_pairs),
        to_edge_index(team_to_team_pairs)
    )

edge_team_match, edge_match_team, edge_team_team = build_team_match_edges(edges)

print("edge_team_match:", edge_team_match.shape)
print("edge_match_team:", edge_match_team.shape)
print("edge_team_team:", edge_team_team.shape)


edge_team_match: (2, 1000)
edge_match_team: (2, 1000)
edge_team_team: (2, 1000)


# Leakage note для prototype graph

In [25]:
# Важно:
#
# Текущий simplified Team-Match graph использует все nodes/edges 1994-2022.
# Это transductive prototype:
# - labels используются только через train mask;
# - но structural information из будущих матчей может быть доступна через graph topology.
#
# Поэтому результаты этого варианта нужно помечать как exploratory.
#
# Leakage-safer следующий шаг:
# - train graph: historical graph up to 2014
# - val graph: graph up to pre-2018 / without val labels
# - test graph: graph up to pre-2022 / without test labels
# - ещё лучше: dynamic graph with edges up to match timestamp


# Создание PyTorch Geometric HeteroData

In [138]:
def create_team_match_heterodata(X_match, model_name_suffix=""):
    if not PYG_AVAILABLE:
        print("PyG is not available. Cannot create HeteroData.")
        return None
    
    data = HeteroData()
    
    data["team"].x = torch.tensor(X_team, dtype=torch.float32)
    data["match"].x = torch.tensor(X_match, dtype=torch.float32)
    data["match"].y = torch.tensor(y_gnn, dtype=torch.long)
    
    data["match"].train_mask = torch.tensor(train_mask_gnn, dtype=torch.bool)
    data["match"].val_mask = torch.tensor(val_mask_gnn, dtype=torch.bool)
    data["match"].test_mask = torch.tensor(test_mask_gnn, dtype=torch.bool)
    
    data["team", "plays", "match"].edge_index = torch.tensor(edge_team_match, dtype=torch.long)
    data["match", "rev_plays", "team"].edge_index = torch.tensor(edge_match_team, dtype=torch.long)
    
    if edge_team_team.shape[1] > 0:
        data["team", "played_against", "team"].edge_index = torch.tensor(edge_team_team, dtype=torch.long)
    
    return data

if PYG_AVAILABLE:
    data_numeric_text = create_team_match_heterodata(X_match_numeric_text_gnn)
    print(data_numeric_text)
    print("Metadata:", data_numeric_text.metadata())


HeteroData(
  team={ x=[70, 2] },
  match={
    x=[500, 133],
    y=[500],
    train_mask=[500],
    val_mask=[500],
    test_mask=[500],
  },
  (team, plays, match)={ edge_index=[2, 1000] },
  (match, rev_plays, team)={ edge_index=[2, 1000] },
  (team, played_against, team)={ edge_index=[2, 1000] }
)
Metadata: (['team', 'match'], [('team', 'plays', 'match'), ('match', 'rev_plays', 'team'), ('team', 'played_against', 'team')])


# Heterogeneous Team-Match GraphSAGE model

In [139]:
if TORCH_AVAILABLE and PYG_AVAILABLE:
    
    class TeamMatchHeteroSAGE(nn.Module):
        def __init__(
            self,
            metadata,
            in_channels_dict,
            hidden_dim=64,
            out_dim=3,
            num_layers=2,
            dropout=0.35
        ):
            super().__init__()
            
            self.metadata = metadata
            self.hidden_dim = hidden_dim
            self.dropout = dropout
            
            self.input_proj = nn.ModuleDict()
            for node_type, in_dim in in_channels_dict.items():
                self.input_proj[node_type] = nn.Sequential(
                    nn.Linear(in_dim, hidden_dim),
                    nn.ReLU(),
                    nn.Dropout(dropout)
                )
            
            self.convs = nn.ModuleList()
            for _ in range(num_layers):
                conv_dict = {}
                for edge_type in metadata[1]:
                    conv_dict[edge_type] = SAGEConv(
                        (-1, -1),
                        hidden_dim
                    )
                self.convs.append(
                    HeteroConv(conv_dict, aggr="sum")
                )
            
            self.classifier = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, out_dim)
            )
        
        def forward(self, x_dict, edge_index_dict):
            x_dict = {
                node_type: self.input_proj[node_type](x)
                for node_type, x in x_dict.items()
            }
            
            for conv in self.convs:
                x_new = conv(x_dict, edge_index_dict)
                
                # Некоторые node types могут не обновиться, если нет incoming edges.
                # Сохраняем старое представление как residual fallback.
                out_dict = {}
                for node_type in x_dict:
                    if node_type in x_new and x_new[node_type] is not None:
                        h = F.relu(x_new[node_type])
                        h = F.dropout(h, p=self.dropout, training=self.training)
                        out_dict[node_type] = h + x_dict[node_type]
                    else:
                        out_dict[node_type] = x_dict[node_type]
                
                x_dict = out_dict
            
            logits = self.classifier(x_dict["match"])
            return logits


# Fallback graph-inspired MLP если PyG недоступен

In [140]:
if TORCH_AVAILABLE:
    
    class MatchFeatureMLP(nn.Module):
        """
        Fallback neural baseline.
        Это не полноценная GNN, а graph-inspired fallback.
        Используется, если torch_geometric недоступен.
        """
        def __init__(self, in_dim, hidden_dim=64, out_dim=3, dropout=0.35):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(in_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, out_dim)
            )
        
        def forward(self, x):
            return self.net(x)


# Evaluation utilities

In [141]:
CLASS_NAMES = ["home_win", "draw", "away_win"]

def compute_metrics(y_true, y_pred, y_proba=None, prefix=""):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    
    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        zero_division=0
    )
    
    metrics = {
        f"accuracy{prefix}": acc,
        f"balanced_accuracy{prefix}": bal_acc,
        f"macro_f1{prefix}": macro_f1,
        f"weighted_f1{prefix}": weighted_f1,
        f"draw_recall{prefix}": recall[1],
        f"draw_f1{prefix}": f1[1],
    }
    
    return metrics


def print_detailed_report(y_true, y_pred, title=""):
    print("=" * 80)
    print(title)
    print("=" * 80)
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred, labels=[0, 1, 2]))
    print()
    print(classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        target_names=CLASS_NAMES,
        zero_division=0
    ))


# Train/eval functions для PyG

In [142]:
if TORCH_AVAILABLE and PYG_AVAILABLE:
    
    def train_pyg_model(
        data,
        model_name="gnn_team_match_numeric_text",
        hidden_dim=64,
        num_layers=2,
        dropout=0.35,
        lr=1e-3,
        weight_decay=1e-4,
        max_epochs=400,
        patience=50,
        grad_clip=1.0,
        verbose=True
    ):
        seed_everything(RANDOM_SEED)
        
        data = data.to(DEVICE)
        
        in_channels_dict = {
            node_type: data[node_type].x.shape[1]
            for node_type in data.node_types
        }
        
        model = TeamMatchHeteroSAGE(
            metadata=data.metadata(),
            in_channels_dict=in_channels_dict,
            hidden_dim=hidden_dim,
            out_dim=3,
            num_layers=num_layers,
            dropout=dropout
        ).to(DEVICE)
        
        weight_tensor = torch.tensor(
            [class_weights.get(i, 1.0) for i in range(3)],
            dtype=torch.float32,
            device=DEVICE
        )
        
        criterion = nn.CrossEntropyLoss(weight=weight_tensor)
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay
        )
        
        train_mask = data["match"].train_mask
        val_mask = data["match"].val_mask
        y = data["match"].y
        
        best_val_macro_f1 = -1
        best_state = None
        best_epoch = -1
        wait = 0
        history = []
        
        for epoch in range(1, max_epochs + 1):
            model.train()
            optimizer.zero_grad()
            
            logits = model(data.x_dict, data.edge_index_dict)
            loss = criterion(logits[train_mask], y[train_mask])
            loss.backward()
            
            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            
            optimizer.step()
            
            # Eval
            model.eval()
            with torch.no_grad():
                logits_eval = model(data.x_dict, data.edge_index_dict)
                proba = F.softmax(logits_eval, dim=1)
                pred = proba.argmax(dim=1)
                
                y_val = y[val_mask].detach().cpu().numpy()
                pred_val = pred[val_mask].detach().cpu().numpy()
                
                val_metrics = compute_metrics(y_val, pred_val, prefix="_val")
                val_macro_f1 = val_metrics["macro_f1_val"]
            
            history.append({
                "epoch": epoch,
                "loss": float(loss.detach().cpu()),
                **val_metrics
            })
            
            if val_macro_f1 > best_val_macro_f1:
                best_val_macro_f1 = val_macro_f1
                best_state = {
                    k: v.detach().cpu().clone()
                    for k, v in model.state_dict().items()
                }
                best_epoch = epoch
                wait = 0
            else:
                wait += 1
            
            if verbose and (epoch == 1 or epoch % 25 == 0):
                print(
                    f"Epoch {epoch:03d} | "
                    f"loss={float(loss.detach().cpu()):.4f} | "
                    f"val_macro_f1={val_macro_f1:.4f} | "
                    f"best={best_val_macro_f1:.4f}@{best_epoch}"
                )
            
            if wait >= patience:
                if verbose:
                    print(f"Early stopping at epoch {epoch}. Best epoch: {best_epoch}")
                break
        
        if best_state is not None:
            model.load_state_dict(best_state)
        
        return model, pd.DataFrame(history), best_epoch, best_val_macro_f1
    
    
    def predict_pyg_model(model, data):
        model.eval()
        data = data.to(DEVICE)
        
        with torch.no_grad():
            logits = model(data.x_dict, data.edge_index_dict)
            proba = F.softmax(logits, dim=1).detach().cpu().numpy()
            pred = proba.argmax(axis=1)
        
        return pred, proba


# Train/eval functions для fallback MLP

In [143]:
if TORCH_AVAILABLE:
    
    def train_mlp_fallback(
        X_match,
        model_name="mlp_fallback_numeric_text",
        hidden_dim=64,
        dropout=0.35,
        lr=1e-3,
        weight_decay=1e-4,
        max_epochs=400,
        patience=50,
        grad_clip=1.0,
        verbose=True
    ):
        seed_everything(RANDOM_SEED)
        
        X = torch.tensor(X_match, dtype=torch.float32, device=DEVICE)
        y = torch.tensor(y_gnn, dtype=torch.long, device=DEVICE)
        
        train_mask = torch.tensor(train_mask_gnn, dtype=torch.bool, device=DEVICE)
        val_mask = torch.tensor(val_mask_gnn, dtype=torch.bool, device=DEVICE)
        
        model = MatchFeatureMLP(
            in_dim=X_match.shape[1],
            hidden_dim=hidden_dim,
            out_dim=3,
            dropout=dropout
        ).to(DEVICE)
        
        weight_tensor = torch.tensor(
            [class_weights.get(i, 1.0) for i in range(3)],
            dtype=torch.float32,
            device=DEVICE
        )
        
        criterion = nn.CrossEntropyLoss(weight=weight_tensor)
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay
        )
        
        best_val_macro_f1 = -1
        best_state = None
        best_epoch = -1
        wait = 0
        history = []
        
        for epoch in range(1, max_epochs + 1):
            model.train()
            optimizer.zero_grad()
            
            logits = model(X)
            loss = criterion(logits[train_mask], y[train_mask])
            loss.backward()
            
            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            
            optimizer.step()
            
            model.eval()
            with torch.no_grad():
                logits_eval = model(X)
                proba = F.softmax(logits_eval, dim=1)
                pred = proba.argmax(dim=1)
                
                y_val = y[val_mask].detach().cpu().numpy()
                pred_val = pred[val_mask].detach().cpu().numpy()
                
                val_metrics = compute_metrics(y_val, pred_val, prefix="_val")
                val_macro_f1 = val_metrics["macro_f1_val"]
            
            history.append({
                "epoch": epoch,
                "loss": float(loss.detach().cpu()),
                **val_metrics
            })
            
            if val_macro_f1 > best_val_macro_f1:
                best_val_macro_f1 = val_macro_f1
                best_state = {
                    k: v.detach().cpu().clone()
                    for k, v in model.state_dict().items()
                }
                best_epoch = epoch
                wait = 0
            else:
                wait += 1
            
            if verbose and (epoch == 1 or epoch % 25 == 0):
                print(
                    f"Epoch {epoch:03d} | "
                    f"loss={float(loss.detach().cpu()):.4f} | "
                    f"val_macro_f1={val_macro_f1:.4f} | "
                    f"best={best_val_macro_f1:.4f}@{best_epoch}"
                )
            
            if wait >= patience:
                if verbose:
                    print(f"Early stopping at epoch {epoch}. Best epoch: {best_epoch}")
                break
        
        if best_state is not None:
            model.load_state_dict(best_state)
        
        return model, pd.DataFrame(history), best_epoch, best_val_macro_f1
    
    
    def predict_mlp_fallback(model, X_match):
        model.eval()
        X = torch.tensor(X_match, dtype=torch.float32, device=DEVICE)
        with torch.no_grad():
            logits = model(X)
            proba = F.softmax(logits, dim=1).detach().cpu().numpy()
            pred = proba.argmax(axis=1)
        return pred, proba


# Функция полного эксперимента

In [144]:
def make_predictions_df(mask, pred, proba, model_name):
    df = matches_gnn.loc[mask, [
        "match_id",
        "tournament_year",
        "home_team_name",
        "away_team_name",
        "target"
    ]].copy()
    
    df = df.rename(columns={"target": "true_target"})
    idx = np.where(mask)[0]
    
    df["pred_target"] = pred[idx]
    df["proba_home_win"] = proba[idx, 0]
    df["proba_draw"] = proba[idx, 1]
    df["proba_away_win"] = proba[idx, 2]
    df["model_name"] = model_name
    
    return df


def evaluate_experiment_predictions(pred, proba, model_name, feature_set):
    y_true = y_gnn
    
    val_idx = val_mask_gnn
    test_idx = test_mask_gnn
    
    val_metrics = compute_metrics(
        y_true[val_idx],
        pred[val_idx],
        proba[val_idx],
        prefix="_val"
    )
    
    test_metrics = compute_metrics(
        y_true[test_idx],
        pred[test_idx],
        proba[test_idx],
        prefix="_test"
    )
    
    result = {
        "model_name": model_name,
        "feature_set": feature_set,
        **val_metrics,
        **test_metrics
    }
    
    print_detailed_report(
        y_true[val_idx],
        pred[val_idx],
        title=f"{model_name} - validation"
    )
    
    print_detailed_report(
        y_true[test_idx],
        pred[test_idx],
        title=f"{model_name} - test"
    )
    
    pred_val_df = make_predictions_df(val_idx, pred, proba, model_name)
    pred_test_df = make_predictions_df(test_idx, pred, proba, model_name)
    
    return result, pred_val_df, pred_test_df


def run_experiment(
    model_name,
    feature_set,
    X_match,
    use_pyg=True,
    hidden_dim=64,
    num_layers=2,
    dropout=0.35,
    lr=1e-3,
    weight_decay=1e-4,
    max_epochs=400,
    patience=50
):
    print("=" * 100)
    print("Running experiment:", model_name)
    print("feature_set:", feature_set)
    print("X_match:", X_match.shape)
    print("=" * 100)
    
    if use_pyg and PYG_AVAILABLE:
        data = create_team_match_heterodata(X_match)
        model, history, best_epoch, best_val = train_pyg_model(
            data=data,
            model_name=model_name,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            dropout=dropout,
            lr=lr,
            weight_decay=weight_decay,
            max_epochs=max_epochs,
            patience=patience,
            verbose=True
        )
        pred, proba = predict_pyg_model(model, data)
    else:
        print("Using fallback MLP instead of PyG GNN.")
        model, history, best_epoch, best_val = train_mlp_fallback(
            X_match=X_match,
            model_name=model_name,
            hidden_dim=hidden_dim,
            dropout=dropout,
            lr=lr,
            weight_decay=weight_decay,
            max_epochs=max_epochs,
            patience=patience,
            verbose=True
        )
        pred, proba = predict_mlp_fallback(model, X_match)
    
    result, pred_val_df, pred_test_df = evaluate_experiment_predictions(
        pred=pred,
        proba=proba,
        model_name=model_name,
        feature_set=feature_set
    )
    
    result["best_epoch"] = best_epoch
    result["best_val_macro_f1_during_training"] = best_val
    result["is_transductive_prototype"] = bool(use_pyg and PYG_AVAILABLE)
    
    return {
        "model": model,
        "history": history,
        "result": result,
        "pred_val": pred_val_df,
        "pred_test": pred_test_df
    }


# Запуск основного Team-Match GNN prototype

In [145]:
experiments = []

main_exp = run_experiment(
    model_name="gnn_team_match_numeric_text" if PYG_AVAILABLE else "mlp_fallback_numeric_text",
    feature_set="numeric_text_pca32",
    X_match=X_match_numeric_text_gnn,
    use_pyg=PYG_AVAILABLE,
    hidden_dim=64,
    num_layers=2,
    dropout=0.35,
    lr=1e-3,
    weight_decay=1e-4,
    max_epochs=400,
    patience=50
)

experiments.append(main_exp)

display(main_exp["history"].tail())
display(pd.DataFrame([main_exp["result"]]))


Running experiment: gnn_team_match_numeric_text
feature_set: numeric_text_pca32
X_match: (500, 133)
Epoch 001 | loss=1.1329 | val_macro_f1=0.2315 | best=0.2315@1
Epoch 025 | loss=0.9292 | val_macro_f1=0.4195 | best=0.4890@6
Epoch 050 | loss=0.8049 | val_macro_f1=0.3604 | best=0.4890@6
Early stopping at epoch 56. Best epoch: 6
gnn_team_match_numeric_text - validation
Confusion matrix:
[[20  4  2]
 [ 6  3  4]
 [ 7  6 12]]

              precision    recall  f1-score   support

    home_win       0.61      0.77      0.68        26
        draw       0.23      0.23      0.23        13
    away_win       0.67      0.48      0.56        25

    accuracy                           0.55        64
   macro avg       0.50      0.49      0.49        64
weighted avg       0.55      0.55      0.54        64

gnn_team_match_numeric_text - test
Confusion matrix:
[[20  2  7]
 [ 6  3  6]
 [ 4  6 10]]

              precision    recall  f1-score   support

    home_win       0.67      0.69      0.68     

,epoch,loss,accuracy_val,balanced_accuracy_val,macro_f1_val,weighted_f1_val,draw_recall_val,draw_f1_val
51,52,0.799378,0.375000,0.365128,0.365128,0.395833,0.307692,0.205128
52,53,0.809879,0.359375,0.351795,0.353079,0.382674,0.307692,0.200000
53,54,0.823064,0.359375,0.351795,0.353079,0.382674,0.307692,0.200000
54,55,0.780114,0.359375,0.351795,0.353079,0.382674,0.307692,0.200000
55,56,0.784141,0.359375,0.351282,0.356885,0.388670,0.307692,0.195122


,model_name,feature_set,accuracy_val,balanced_accuracy_val,macro_f1_val,weighted_f1_val,draw_recall_val,draw_f1_val,accuracy_test,balanced_accuracy_test,macro_f1_test,weighted_f1_test,draw_recall_test,draw_f1_test,best_epoch,best_val_macro_f1_during_training,is_transductive_prototype
0,gnn_team_match_numeric_text,numeric_text_pca32,0.546875,0.493333,0.488958,0.540322,0.230769,0.230769,0.515625,0.463218,0.457951,0.506639,0.2,0.230769,6,0.488958,True


# GNN ablation: numeric only

In [146]:
exp_numeric_only = run_experiment(
    model_name="gnn_match_numeric_only" if PYG_AVAILABLE else "mlp_fallback_numeric_only",
    feature_set="numeric_only",
    X_match=X_match_numeric_only_gnn,
    use_pyg=PYG_AVAILABLE,
    hidden_dim=64,
    num_layers=2,
    dropout=0.40,
    lr=1e-3,
    weight_decay=1e-4,
    max_epochs=400,
    patience=50
)

experiments.append(exp_numeric_only)

display(pd.DataFrame([exp_numeric_only["result"]]))


Running experiment: gnn_match_numeric_only
feature_set: numeric_only
X_match: (500, 69)
Epoch 001 | loss=1.1076 | val_macro_f1=0.3722 | best=0.3722@1
Epoch 025 | loss=0.9288 | val_macro_f1=0.4281 | best=0.4733@10
Epoch 050 | loss=0.8532 | val_macro_f1=0.4943 | best=0.4943@50
Epoch 075 | loss=0.7668 | val_macro_f1=0.4593 | best=0.5054@52
Epoch 100 | loss=0.7288 | val_macro_f1=0.4574 | best=0.5054@52
Early stopping at epoch 102. Best epoch: 52
gnn_match_numeric_only - validation
Confusion matrix:
[[14  3  9]
 [ 4  3  6]
 [ 1  5 19]]

              precision    recall  f1-score   support

    home_win       0.74      0.54      0.62        26
        draw       0.27      0.23      0.25        13
    away_win       0.56      0.76      0.64        25

    accuracy                           0.56        64
   macro avg       0.52      0.51      0.51        64
weighted avg       0.57      0.56      0.56        64

gnn_match_numeric_only - test
Confusion matrix:
[[15  6  8]
 [ 3  5  7]
 [ 4  4 1

,model_name,feature_set,accuracy_val,balanced_accuracy_val,macro_f1_val,weighted_f1_val,draw_recall_val,draw_f1_val,accuracy_test,balanced_accuracy_test,macro_f1_test,weighted_f1_test,draw_recall_test,draw_f1_test,best_epoch,best_val_macro_f1_during_training,is_transductive_prototype
0,gnn_match_numeric_only,numeric_only,0.5625,0.509744,0.50543,0.555148,0.230769,0.25,0.5,0.483525,0.477402,0.504244,0.333333,0.333333,52,0.50543,True


# GNN ablation: numeric + text + meta/stage

In [147]:
exp_numeric_text_extra = run_experiment(
    model_name="gnn_team_match_numeric_text_stage" if USE_STAGE_FEATURES else "gnn_team_match_numeric_text_meta",
    feature_set="numeric_text_pca32_extra",
    X_match=X_match_numeric_text_extra_gnn,
    use_pyg=PYG_AVAILABLE,
    hidden_dim=64,
    num_layers=2,
    dropout=0.40,
    lr=1e-3,
    weight_decay=1e-4,
    max_epochs=400,
    patience=50
)

experiments.append(exp_numeric_text_extra)

display(pd.DataFrame([exp_numeric_text_extra["result"]]))


Running experiment: gnn_team_match_numeric_text_meta
feature_set: numeric_text_pca32_extra
X_match: (500, 138)
Epoch 001 | loss=1.1276 | val_macro_f1=0.2269 | best=0.2269@1
Epoch 025 | loss=0.9281 | val_macro_f1=0.4606 | best=0.4606@23
Epoch 050 | loss=0.8281 | val_macro_f1=0.4836 | best=0.5197@44
Epoch 075 | loss=0.7455 | val_macro_f1=0.4345 | best=0.5197@44
Early stopping at epoch 94. Best epoch: 44
gnn_team_match_numeric_text_meta - validation
Confusion matrix:
[[16  1  9]
 [ 5  3  5]
 [ 1  6 18]]

              precision    recall  f1-score   support

    home_win       0.73      0.62      0.67        26
        draw       0.30      0.23      0.26        13
    away_win       0.56      0.72      0.63        25

    accuracy                           0.58        64
   macro avg       0.53      0.52      0.52        64
weighted avg       0.58      0.58      0.57        64

gnn_team_match_numeric_text_meta - test
Confusion matrix:
[[17  5  7]
 [ 4  3  8]
 [ 5  3 12]]

              pr

,model_name,feature_set,accuracy_val,balanced_accuracy_val,macro_f1_val,weighted_f1_val,draw_recall_val,draw_f1_val,accuracy_test,balanced_accuracy_test,macro_f1_test,weighted_f1_test,draw_recall_test,draw_f1_test,best_epoch,best_val_macro_f1_during_training,is_transductive_prototype
0,gnn_team_match_numeric_text_meta,numeric_text_pca32_extra,0.578125,0.522051,0.519705,0.570533,0.230769,0.26087,0.5,0.462069,0.453196,0.493775,0.2,0.230769,44,0.519705,True


# Если GNN переобучается: smaller model ablation

In [148]:
exp_small = run_experiment(
    model_name="gnn_team_match_numeric_text_small" if PYG_AVAILABLE else "mlp_fallback_numeric_text_small",
    feature_set="numeric_text_pca32",
    X_match=X_match_numeric_text_gnn,
    use_pyg=PYG_AVAILABLE,
    hidden_dim=32,
    num_layers=2,
    dropout=0.50,
    lr=1e-3,
    weight_decay=3e-4,
    max_epochs=400,
    patience=50
)

experiments.append(exp_small)

display(pd.DataFrame([exp_small["result"]]))


Running experiment: gnn_team_match_numeric_text_small
feature_set: numeric_text_pca32
X_match: (500, 133)
Epoch 001 | loss=1.1854 | val_macro_f1=0.2286 | best=0.2286@1
Epoch 025 | loss=1.0465 | val_macro_f1=0.3680 | best=0.4672@7
Epoch 050 | loss=0.9713 | val_macro_f1=0.4228 | best=0.4819@43
Epoch 075 | loss=0.8990 | val_macro_f1=0.4477 | best=0.4828@71
Epoch 100 | loss=0.8523 | val_macro_f1=0.3926 | best=0.4828@71
Early stopping at epoch 121. Best epoch: 71
gnn_team_match_numeric_text_small - validation
Confusion matrix:
[[14  5  7]
 [ 4  4  5]
 [ 3  7 15]]

              precision    recall  f1-score   support

    home_win       0.67      0.54      0.60        26
        draw       0.25      0.31      0.28        13
    away_win       0.56      0.60      0.58        25

    accuracy                           0.52        64
   macro avg       0.49      0.48      0.48        64
weighted avg       0.54      0.52      0.52        64

gnn_team_match_numeric_text_small - test
Confusion ma

,model_name,feature_set,accuracy_val,balanced_accuracy_val,macro_f1_val,weighted_f1_val,draw_recall_val,draw_f1_val,accuracy_test,balanced_accuracy_test,macro_f1_test,weighted_f1_test,draw_recall_test,draw_f1_test,best_epoch,best_val_macro_f1_during_training,is_transductive_prototype
0,gnn_team_match_numeric_text_small,numeric_text_pca32,0.515625,0.482051,0.482843,0.523416,0.307692,0.275862,0.5625,0.534674,0.52986,0.564361,0.333333,0.357143,71,0.482843,True


# Сохранение GNN results

In [149]:
gnn_results = pd.DataFrame([e["result"] for e in experiments])

# Приводим к требуемому набору колонок + дополнительные diagnostic columns
required_result_cols = [
    "model_name",
    "feature_set",
    "accuracy_val",
    "balanced_accuracy_val",
    "macro_f1_val",
    "weighted_f1_val",
    "draw_recall_val",
    "draw_f1_val",
    "accuracy_test",
    "balanced_accuracy_test",
    "macro_f1_test",
    "weighted_f1_test",
    "draw_recall_test",
    "draw_f1_test",
]

extra_cols = [c for c in gnn_results.columns if c not in required_result_cols]
gnn_results = gnn_results[required_result_cols + extra_cols]

gnn_results_path = DATA_DIR / "gnn_results.csv"
gnn_results.to_csv(gnn_results_path, index=False)

print("Saved:", gnn_results_path)
display(gnn_results.sort_values("macro_f1_val", ascending=False))


Saved: SNA/data/processed/gnn_results.csv


,model_name,feature_set,accuracy_val,balanced_accuracy_val,macro_f1_val,weighted_f1_val,draw_recall_val,draw_f1_val,accuracy_test,balanced_accuracy_test,macro_f1_test,weighted_f1_test,draw_recall_test,draw_f1_test,best_epoch,best_val_macro_f1_during_training,is_transductive_prototype
2,gnn_team_match_numeric_text_meta,numeric_text_pca32_extra,0.578125,0.522051,0.519705,0.570533,0.230769,0.260870,0.500000,0.462069,0.453196,0.493775,0.200000,0.230769,44,0.519705,True
1,gnn_match_numeric_only,numeric_only,0.562500,0.509744,0.505430,0.555148,0.230769,0.250000,0.500000,0.483525,0.477402,0.504244,0.333333,0.333333,52,0.505430,True
0,gnn_team_match_numeric_text,numeric_text_pca32,0.546875,0.493333,0.488958,0.540322,0.230769,0.230769,0.515625,0.463218,0.457951,0.506639,0.200000,0.230769,6,0.488958,True
3,gnn_team_match_numeric_text_small,numeric_text_pca32,0.515625,0.482051,0.482843,0.523416,0.307692,0.275862,0.562500,0.534674,0.529860,0.564361,0.333333,0.357143,71,0.482843,True


# Сохранение predictions лучшей GNN по validation macro-F1

In [150]:
best_idx = gnn_results["macro_f1_val"].astype(float).idxmax()
best_model_name = gnn_results.loc[best_idx, "model_name"]

print("Best model by validation macro-F1:", best_model_name)

best_exp = None
for e in experiments:
    if e["result"]["model_name"] == best_model_name:
        best_exp = e
        break

if best_exp is None:
    raise ValueError("Could not find best experiment object.")

pred_val_path = DATA_DIR / "gnn_predictions_val.csv"
pred_test_path = DATA_DIR / "gnn_predictions_test.csv"

best_exp["pred_val"].to_csv(pred_val_path, index=False)
best_exp["pred_test"].to_csv(pred_test_path, index=False)

print("Saved:", pred_val_path)
print("Saved:", pred_test_path)

display(best_exp["pred_val"].head())
display(best_exp["pred_test"].head())


Best model by validation macro-F1: gnn_team_match_numeric_text_meta
Saved: SNA/data/processed/gnn_predictions_val.csv
Saved: SNA/data/processed/gnn_predictions_test.csv


,match_id,tournament_year,home_team_name,away_team_name,true_target,pred_target,proba_home_win,proba_draw,proba_away_win,model_name
372,match_3ee0a05214bf,2018,Russia,Saudi Arabia,0,0,0.483410,0.430629,0.085961,gnn_team_match_numeric_text_meta
373,match_0a7804398319,2018,Portugal,Spain,1,1,0.178019,0.412311,0.409671,gnn_team_match_numeric_text_meta
374,match_53a59c1e0e89,2018,Morocco,IR Iran,2,2,0.158809,0.418781,0.422409,gnn_team_match_numeric_text_meta
375,match_b0d45a81a705,2018,Egypt,Uruguay,2,2,0.019934,0.230127,0.749939,gnn_team_match_numeric_text_meta
376,match_3bcb591a98d4,2018,Argentina,Iceland,1,0,0.556563,0.295449,0.147988,gnn_team_match_numeric_text_meta


,match_id,tournament_year,home_team_name,away_team_name,true_target,pred_target,proba_home_win,proba_draw,proba_away_win,model_name
436,match_eba59d5d01d0,2022,Qatar,Ecuador,2,1,0.165422,0.456676,0.377901,gnn_team_match_numeric_text_meta
437,match_0a8f5c4a4766,2022,United States,Wales,1,0,0.389251,0.372515,0.238235,gnn_team_match_numeric_text_meta
438,match_80a4ca985d2c,2022,Senegal,Netherlands,2,2,0.082406,0.353548,0.564046,gnn_team_match_numeric_text_meta
439,match_a64cba8917b3,2022,England,IR Iran,0,0,0.485733,0.360151,0.154117,gnn_team_match_numeric_text_meta
440,match_281c5a33aca4,2022,Denmark,Tunisia,1,0,0.456859,0.385343,0.157797,gnn_team_match_numeric_text_meta


# Сравнение с baseline models

In [151]:
if baseline_results is not None and not baseline_results.empty:
    key_baselines = [
        "ranking_compact_rf",
        "combined_numeric_rf_safe",
        "numeric_text_rf"
    ]
    
    # Пытаемся найти колонку с model name
    possible_model_cols = ["model_name", "model", "name"]
    baseline_model_col = next(
        (c for c in possible_model_cols if c in baseline_results.columns),
        None
    )
    
    if baseline_model_col is None:
        print("Cannot infer baseline model name column.")
        display(baseline_results.head())
    else:
        baseline_key = baseline_results[
            baseline_results[baseline_model_col].isin(key_baselines)
        ].copy()
        
        print("Key baseline rows:")
        display(baseline_key)
        
        print("GNN rows:")
        display(gnn_results.sort_values("macro_f1_val", ascending=False))
else:
    print("No baseline results loaded.")


Key baseline rows:


,model_name,feature_set,accuracy_val,balanced_accuracy_val,macro_f1_val,weighted_f1_val,draw_recall_val,draw_f1_val,accuracy_test,balanced_accuracy_test,macro_f1_test,weighted_f1_test,draw_recall_test,draw_f1_test
0,ranking_compact_rf,ranking_compact,0.593750,0.573846,0.565187,0.596776,0.461538,0.400000,0.500000,0.477969,0.469039,0.497771,0.266667,0.275862
1,numeric_text_rf,combined_numeric_plus_text_embeddings,0.640625,0.562051,0.546268,0.606456,0.153846,0.235294,0.578125,0.551724,0.553846,0.576923,0.400000,0.461538
2,combined_numeric_rf_safe,combined_numeric,0.531250,0.483077,0.489872,0.547501,0.230769,0.193548,0.546875,0.550192,0.538936,0.552258,0.533333,0.457143


GNN rows:


,model_name,feature_set,accuracy_val,balanced_accuracy_val,macro_f1_val,weighted_f1_val,draw_recall_val,draw_f1_val,accuracy_test,balanced_accuracy_test,macro_f1_test,weighted_f1_test,draw_recall_test,draw_f1_test,best_epoch,best_val_macro_f1_during_training,is_transductive_prototype
2,gnn_team_match_numeric_text_meta,numeric_text_pca32_extra,0.578125,0.522051,0.519705,0.570533,0.230769,0.260870,0.500000,0.462069,0.453196,0.493775,0.200000,0.230769,44,0.519705,True
1,gnn_match_numeric_only,numeric_only,0.562500,0.509744,0.505430,0.555148,0.230769,0.250000,0.500000,0.483525,0.477402,0.504244,0.333333,0.333333,52,0.505430,True
0,gnn_team_match_numeric_text,numeric_text_pca32,0.546875,0.493333,0.488958,0.540322,0.230769,0.230769,0.515625,0.463218,0.457951,0.506639,0.200000,0.230769,6,0.488958,True
3,gnn_team_match_numeric_text_small,numeric_text_pca32,0.515625,0.482051,0.482843,0.523416,0.307692,0.275862,0.562500,0.534674,0.529860,0.564361,0.333333,0.357143,71,0.482843,True


# Унифицированная таблица сравнения

In [152]:
def normalize_baseline_results_for_comparison(baseline_results):
    if baseline_results is None or baseline_results.empty:
        return pd.DataFrame()
    
    possible_model_cols = ["model_name", "model", "name"]
    model_col = next((c for c in possible_model_cols if c in baseline_results.columns), None)
    if model_col is None:
        return pd.DataFrame()
    
    df = baseline_results.copy()
    df = df.rename(columns={model_col: "model_name"})
    
    # Возможные варианты имён колонок
    rename_map = {}
    
    candidates = {
        "accuracy_val": ["accuracy_val", "val_accuracy", "acc_val"],
        "balanced_accuracy_val": ["balanced_accuracy_val", "val_balanced_accuracy"],
        "macro_f1_val": ["macro_f1_val", "val_macro_f1"],
        "weighted_f1_val": ["weighted_f1_val", "val_weighted_f1"],
        "draw_recall_val": ["draw_recall_val", "val_draw_recall"],
        "draw_f1_val": ["draw_f1_val", "val_draw_f1"],
        "accuracy_test": ["accuracy_test", "test_accuracy", "acc_test"],
        "balanced_accuracy_test": ["balanced_accuracy_test", "test_balanced_accuracy"],
        "macro_f1_test": ["macro_f1_test", "test_macro_f1"],
        "weighted_f1_test": ["weighted_f1_test", "test_weighted_f1"],
        "draw_recall_test": ["draw_recall_test", "test_draw_recall"],
        "draw_f1_test": ["draw_f1_test", "test_draw_f1"],
    }
    
    for standard, opts in candidates.items():
        for opt in opts:
            if opt in df.columns:
                rename_map[opt] = standard
                break
    
    df = df.rename(columns=rename_map)
    
    keep = ["model_name"] + [c for c in required_result_cols if c != "model_name" and c in df.columns]
    df = df[keep].copy()
    df["source"] = "baseline"
    
    return df

baseline_comp = normalize_baseline_results_for_comparison(baseline_results)

if not baseline_comp.empty:
    key_baselines = [
        "ranking_compact_rf",
        "combined_numeric_rf_safe",
        "numeric_text_rf"
    ]
    baseline_comp = baseline_comp[baseline_comp["model_name"].isin(key_baselines)]
    
gnn_comp = gnn_results[required_result_cols].copy()
gnn_comp["source"] = "gnn"

comparison = pd.concat([baseline_comp, gnn_comp], axis=0, ignore_index=True)

sort_col = "macro_f1_val" if "macro_f1_val" in comparison.columns else None
if sort_col:
    comparison = comparison.sort_values(sort_col, ascending=False)

display(comparison)

comparison_path = DATA_DIR / "gnn_vs_baseline_comparison.csv"
comparison.to_csv(comparison_path, index=False)
print("Saved:", comparison_path)


,model_name,feature_set,accuracy_val,balanced_accuracy_val,macro_f1_val,weighted_f1_val,draw_recall_val,draw_f1_val,accuracy_test,balanced_accuracy_test,macro_f1_test,weighted_f1_test,draw_recall_test,draw_f1_test,source
0,ranking_compact_rf,ranking_compact,0.593750,0.573846,0.565187,0.596776,0.461538,0.400000,0.500000,0.477969,0.469039,0.497771,0.266667,0.275862,baseline
1,numeric_text_rf,combined_numeric_plus_text_embeddings,0.640625,0.562051,0.546268,0.606456,0.153846,0.235294,0.578125,0.551724,0.553846,0.576923,0.400000,0.461538,baseline
5,gnn_team_match_numeric_text_meta,numeric_text_pca32_extra,0.578125,0.522051,0.519705,0.570533,0.230769,0.260870,0.500000,0.462069,0.453196,0.493775,0.200000,0.230769,gnn
4,gnn_match_numeric_only,numeric_only,0.562500,0.509744,0.505430,0.555148,0.230769,0.250000,0.500000,0.483525,0.477402,0.504244,0.333333,0.333333,gnn
2,combined_numeric_rf_safe,combined_numeric,0.531250,0.483077,0.489872,0.547501,0.230769,0.193548,0.546875,0.550192,0.538936,0.552258,0.533333,0.457143,baseline
3,gnn_team_match_numeric_text,numeric_text_pca32,0.546875,0.493333,0.488958,0.540322,0.230769,0.230769,0.515625,0.463218,0.457951,0.506639,0.200000,0.230769,gnn
6,gnn_team_match_numeric_text_small,numeric_text_pca32,0.515625,0.482051,0.482843,0.523416,0.307692,0.275862,0.562500,0.534674,0.529860,0.564361,0.333333,0.357143,gnn


Saved: SNA/data/processed/gnn_vs_baseline_comparison.csv


# GNN ablation план

In [153]:
gnn_ablation_plan = pd.DataFrame([
    {
        "experiment": "gnn_match_numeric_only",
        "description": "Match node numeric features only; Team-Match message passing",
        "purpose": "Проверить, помогает ли graph structure без текста"
    },
    {
        "experiment": "gnn_match_numeric_text",
        "description": "Match numeric + text PCA embeddings",
        "purpose": "Проверить вклад leakage-safe Guardian text embeddings"
    },
    {
        "experiment": "gnn_team_match_numeric_text",
        "description": "Simplified Team-Match heterograph with numeric+text match nodes and structural team features",
        "purpose": "Основной prototype"
    },
    {
        "experiment": "gnn_team_match_numeric_text_stage",
        "description": "Добавить stage/text meta features",
        "purpose": "Проверить stage context и text availability"
    },
    {
        "experiment": "gnn_full_hetero",
        "description": "Team, Match, Tournament, Stadium, Referee, TeamTournament, Manager",
        "purpose": "Расширение после стабилизации simplified graph"
    }
])

display(gnn_ablation_plan)


,experiment,description,purpose
0,gnn_match_numeric_only,Match node numeric features only; Team-Match m...,"Проверить, помогает ли graph structure без текста"
1,gnn_match_numeric_text,Match numeric + text PCA embeddings,Проверить вклад leakage-safe Guardian text emb...
2,gnn_team_match_numeric_text,Simplified Team-Match heterograph with numeric...,Основной prototype
3,gnn_team_match_numeric_text_stage,Добавить stage/text meta features,Проверить stage context и text availability
4,gnn_full_hetero,"Team, Match, Tournament, Stadium, Referee, Tea...",Расширение после стабилизации simplified graph


# Итоговые рекомендации

In [154]:
best_gnn = gnn_results.sort_values("macro_f1_val", ascending=False).iloc[0]

print("Best GNN/fallback by validation macro-F1:")
display(best_gnn.to_frame().T)

print("""
Рекомендации по интерпретации:

1. Основная метрика выбора модели:
   - validation macro-F1.
   Test 2022 не использовать для выбора модели.

2. Дополнительные ключевые метрики:
   - draw recall;
   - draw F1;
   - balanced accuracy.
   Draw class важен, потому что он миноритарный и сложный.

3. Если GNN хуже numeric_text_rf:
   - это нормально для small-data football task;
   - graph topology в transductive form может быть недостаточно информативной;
   - random forest на tabular/text признаках часто сильнее на 500 матчах.

4. Если train/val сильно расходятся:
   - уменьшить hidden_dim до 32;
   - dropout 0.45-0.55;
   - weight_decay 3e-4 или 1e-3;
   - text PCA 16/32 вместо 64/384;
   - попробовать MLP на тех же features как sanity check.

5. Если GNN улучшил validation macro-F1:
   - проверить draw F1/recall;
   - проверить confusion matrix;
   - затем делать leakage-safer snapshot graph.

6. Full heterograph стоит делать только после:
   - стабильного simplified Team-Match результата;
   - проверки leakage;
   - ablation по relations.

7. Для финального отчёта:
   - transductive GNN обозначить как exploratory;
   - финальный benchmark лучше делать на time-aware graph.
""")


Best GNN/fallback by validation macro-F1:


,model_name,feature_set,accuracy_val,balanced_accuracy_val,macro_f1_val,weighted_f1_val,draw_recall_val,draw_f1_val,accuracy_test,balanced_accuracy_test,macro_f1_test,weighted_f1_test,draw_recall_test,draw_f1_test,best_epoch,best_val_macro_f1_during_training,is_transductive_prototype
2,gnn_team_match_numeric_text_meta,numeric_text_pca32_extra,0.578125,0.522051,0.519705,0.570533,0.230769,0.26087,0.5,0.462069,0.453196,0.493775,0.2,0.230769,44,0.519705,True



Рекомендации по интерпретации:

1. Основная метрика выбора модели:
   - validation macro-F1.
   Test 2022 не использовать для выбора модели.

2. Дополнительные ключевые метрики:
   - draw recall;
   - draw F1;
   - balanced accuracy.
   Draw class важен, потому что он миноритарный и сложный.

3. Если GNN хуже numeric_text_rf:
   - это нормально для small-data football task;
   - graph topology в transductive form может быть недостаточно информативной;
   - random forest на tabular/text признаках часто сильнее на 500 матчах.

4. Если train/val сильно расходятся:
   - уменьшить hidden_dim до 32;
   - dropout 0.45-0.55;
   - weight_decay 3e-4 или 1e-3;
   - text PCA 16/32 вместо 64/384;
   - попробовать MLP на тех же features как sanity check.

5. Если GNN улучшил validation macro-F1:
   - проверить draw F1/recall;
   - проверить confusion matrix;
   - затем делать leakage-safer snapshot graph.

6. Full heterograph стоит делать только после:
   - стабильного simplified Team-Match результ